# 2.3 · Transports: stdio and Streamable HTTP

**传输层：stdio 与 Streamable HTTP**

> **After this lesson you can:** Frame MCP messages correctly over stdio and Streamable HTTP, set and validate the required HTTP headers, secure a local HTTP server, and handle cancellation and older transport versions.

Official reading: [Spec: Transports overview](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/index) · [Spec: stdio](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/stdio) · [Spec: Streamable HTTP](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/streamable-http)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. stdio framing by hand: pipelining, stderr and a broken frame
A server that logs to **stderr** while it works. We pipeline two requests in a single write and watch the replies arrive **by id, out of order**, with stdout and stderr read separately.

In [ ]:
import json
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
import time
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "stdio_server.py").write_text(textwrap.dedent('''
    import sys
    import anyio
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    mcp = MCPServer("stdio-lab", log_level="ERROR")

    @mcp.tool()
    def add(a: int, b: int) -> int:
        """Add two integers."""
        return a + b

    @mcp.tool()
    async def slow(steps: int, ctx: Context) -> str:
        """Work for a while, reporting progress."""
        for i in range(steps):
            print(f"[server log] slow step {i + 1}/{steps}", file=sys.stderr, flush=True)   # stderr: fine
            await ctx.report_progress(i + 1, steps)
            await anyio.sleep(0.3)
        return "done"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))

ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {}}
proc = subprocess.Popen([sys.executable, str(workdir / "stdio_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
stdout_q, stderr_q = queue.Queue(), queue.Queue()
threading.Thread(target=lambda: [stdout_q.put(l) for l in proc.stdout], daemon=True).start()
threading.Thread(target=lambda: [stderr_q.put(l) for l in proc.stderr], daemon=True).start()


def write_lines(*messages):
    data = "".join(json.dumps(m) + "\n" for m in messages)     # compact JSON, one per line
    proc.stdin.write(data)
    proc.stdin.flush()


def read_stdout(timeout=3.0):
    try:
        return json.loads(stdout_q.get(timeout=timeout))
    except queue.Empty:
        return None


write_lines(
    {"jsonrpc": "2.0", "id": "slow-1", "method": "tools/call",
     "params": {"name": "slow", "arguments": {"steps": 3}, "_meta": {**ENVELOPE, "progressToken": "p"}}},
    {"jsonrpc": "2.0", "id": "fast-2", "method": "tools/call",
     "params": {"name": "add", "arguments": {"a": 2, "b": 40}, "_meta": ENVELOPE}},
)
start = time.time()
for _ in range(5):
    msg = read_stdout(timeout=10)
    label = f"response id={msg['id']}" if "id" in msg else f"{msg['method']} {msg['params'].get('progress')}"
    print(f"stdout +{time.time() - start:.2f}s  {label}")
time.sleep(0.1)
while not stderr_q.empty():
    print("stderr          ", stderr_q.get().rstrip())

`fast-2` was written second but answered first. The client matches answers by `id`, not by order. Now break the framing: send the same request **pretty-printed** (embedded newlines).

In [ ]:
pretty = json.dumps({"jsonrpc": "2.0", "id": 3, "method": "tools/list", "params": {"_meta": ENVELOPE}}, indent=2)
print(f"pretty-printed message spans {pretty.count(chr(10)) + 1} lines")
proc.stdin.write(pretty + "\n")
proc.stdin.flush()
print("reply:", read_stdout(timeout=2))

write_lines({"jsonrpc": "2.0", "id": 4, "method": "tools/list", "params": {"_meta": ENVELOPE}})
print("compact reply has", len(read_stdout()["result"]["tools"]), "tools - the server survived, but request 3 was lost")

`json.dumps` without `indent` never emits a raw newline: newlines *inside* strings are escaped as `\n`, so arbitrary text is safe in one frame.

### 2. stdio cancellation and shutdown
Cancel a running request with `notifications/cancelled`, then shut down gracefully by closing stdin.

In [ ]:
write_lines({"jsonrpc": "2.0", "id": "long-5", "method": "tools/call",
             "params": {"name": "slow", "arguments": {"steps": 20}, "_meta": {**ENVELOPE, "progressToken": "q"}}})
print("first message:", read_stdout(timeout=10)["params"])
write_lines({"jsonrpc": "2.0", "method": "notifications/cancelled",
             "params": {"requestId": "long-5", "reason": "user pressed stop"}})
print("after cancel :", read_stdout(timeout=2), "<- no response is sent for a cancelled request")

t = time.time()
proc.stdin.close()                                   # step 1: close the input stream
code = proc.wait(timeout=5)                          # step 2: wait (step 3 would be terminate/kill)
print(f"server exited with code {code} {time.time() - t:.2f}s after EOF")
time.sleep(0.1)
log_lines = []
while not stderr_q.empty():
    log_lines.append(stderr_q.get().strip())
print("stderr shows the cancelled work stopped early:", [l for l in log_lines if "/20" in l])

### 3. Streamable HTTP: one endpoint, raw POSTs
Run an HTTP server on `127.0.0.1` in a thread. A small ASGI wrapper **records every request's headers** so we can see exactly what clients send. One tool declares an `x-mcp-header` parameter.

In [ ]:
import logging
import socket
from typing import Annotated

import anyio
import httpx
import uvicorn
from pydantic import Field
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

web_server = MCPServer("http-lab")


@web_server.tool()
def run_sql(region: Annotated[str, Field(description="Region to run in", json_schema_extra={"x-mcp-header": "Region"})],
            query: str) -> str:
    """Run a (pretend) SQL query in a region."""
    return f"[{region}] {query} -> 42 rows"


@web_server.tool()
async def slow(steps: int, ctx: Context) -> str:
    """Work for a while, reporting progress."""
    try:
        for i in range(steps):
            await ctx.report_progress(i + 1, steps, f"step {i + 1}")
            await anyio.sleep(0.2)
    except BaseException as exc:            # cancellation arrives as an exception in the handler
        server_events.append(f"slow() interrupted by {type(exc).__name__}")
        raise
    return "done"


for noisy in ("httpx", "httpx2", "mcp", "uvicorn"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

server_events, seen_headers = [], []
mcp_app = web_server.streamable_http_app()          # binds DNS-rebinding protection for 127.0.0.1


async def recorder(scope, receive, send):
    if scope["type"] == "http":
        seen_headers.append({"method": scope["method"],
                             **{k.decode(): v.decode() for k, v in scope["headers"] if k.startswith(b"mcp")}})
    await mcp_app(scope, receive, send)


sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()
uv = uvicorn.Server(uvicorn.Config(recorder, host="127.0.0.1", port=port, log_level="warning", lifespan="on"))
threading.Thread(target=uv.run, daemon=True).start()
while not uv.started:
    time.sleep(0.05)
URL = f"http://127.0.0.1:{port}/mcp"
web = httpx.Client(trust_env=False, timeout=10)
HEADERS = {"Accept": "application/json, text/event-stream", "Content-Type": "application/json",
           "MCP-Protocol-Version": "2026-07-28"}


def tools_call(name, arguments, id_=1, meta=ENVELOPE):
    return {"jsonrpc": "2.0", "id": id_, "method": "tools/call",
            "params": {"name": name, "arguments": arguments, "_meta": meta}}


r = web.post(URL, json=tools_call("run_sql", {"region": "us-west1", "query": "select 1"}),
             headers={**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "run_sql", "Mcp-Param-Region": "us-west1"})
print(r.status_code, r.headers["content-type"])
print(json.dumps(r.json(), indent=2)[:600])

### 4. JSON or SSE? Let the request decide
The same `slow` tool is answered as JSON when nobody asked for progress, and as an SSE stream when the request carries a `progressToken`. This SDK only commits to SSE once the handler actually emits a notification.

In [ ]:
def parse_sse(text):
    return [json.loads(line[len("data: "):]) for line in text.splitlines() if line.startswith("data: ")]


h = {**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "slow"}
r = web.post(URL, json=tools_call("slow", {"steps": 2}), headers=h)
print("no progressToken  ->", r.status_code, r.headers["content-type"])

r = web.post(URL, json=tools_call("slow", {"steps": 3}, meta={**ENVELOPE, "progressToken": "tok-1"}), headers=h)
print("with progressToken->", r.status_code, r.headers["content-type"],
      "| X-Accel-Buffering:", r.headers.get("x-accel-buffering"))
for event in parse_sse(r.text):
    print("   SSE event:", event.get("method") or f"final response id={event['id']}",
          event.get("params", {}).get("message", ""))

### 5. Header validation: break every header
Each request below is wrong in exactly one way. All of them except the last two are rejected **before** any tool runs.

In [ ]:
good_body = tools_call("run_sql", {"region": "us-west1", "query": "select 1"})
base = {**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "run_sql", "Mcp-Param-Region": "us-west1"}
cases = {
    "all correct": base,
    "Mcp-Method lies": {**base, "Mcp-Method": "tools/list"},
    "Mcp-Name lies (WAF bypass attempt)": {**base, "Mcp-Name": "get_weather"},
    "Mcp-Name missing": {k: v for k, v in base.items() if k != "Mcp-Name"},
    "Mcp-Param-Region lies": {**base, "Mcp-Param-Region": "eu-west1"},
    "Mcp-Param-Region missing": {k: v for k, v in base.items() if k != "Mcp-Param-Region"},
    "Accept lacks text/event-stream": {**base, "Accept": "application/json"},
    "header NAMES are case-insensitive": {k.upper(): v for k, v in base.items()},
}
old_body = tools_call("run_sql", {"region": "us-west1", "query": "select 1"},
                      meta={**ENVELOPE, "io.modelcontextprotocol/protocolVersion": "2025-11-25"})
for label, headers in cases.items():
    r = web.post(URL, json=good_body, headers=headers)
    detail = r.json().get("error", {}) if r.headers.get("content-type", "").startswith("application/json") else {}
    print(f"{label:<36} HTTP {r.status_code}  {detail.get('code', '')} {detail.get('message', '')[:70]}")

r = web.post(URL, json=old_body, headers=base)          # header says 2026-07-28, body says 2025-11-25
print(f"{'body version != header':<36} HTTP {r.status_code}  {r.json()['error']['code']} {r.json()['error']['message'][:70]}")

The `MCP-Protocol-Version` check runs before the supported-version check, so a client that contradicts itself hears `-32020`, not `-32022`. One quirk: this dual-era SDK picks the *era* from the header, so a request whose **header** names an old version is handed to the legacy transport instead (you will see that in section 8).

Non-ASCII values travel in the base64 sentinel form. The SDK exposes the same encoder clients use:

In [ ]:
from mcp.shared.inbound import encode_header_value

for value in ["us-west1", "Zürich", " padded ", "=?base64?literal?="]:
    print(f"{value!r:<22} -> Mcp-Param-Region: {encode_header_value(value)}")

r = web.post(URL, json=tools_call("run_sql", {"region": "Zürich", "query": "select 1"}),
             headers={**base, "Mcp-Param-Region": encode_header_value("Zürich")})
print("\nencoded header accepted:", r.status_code, r.json()["result"]["content"][0]["text"])

### 6. Origin and Host checks against DNS rebinding
Because the app was built for host `127.0.0.1`, the SDK enabled DNS-rebinding protection with localhost-only allowlists. A page on `evil.example` that rebinds its name to 127.0.0.1 still sends its own `Origin` and `Host`:

In [ ]:
list_body = {"jsonrpc": "2.0", "id": 9, "method": "tools/list", "params": {"_meta": ENVELOPE}}
lh = {**HEADERS, "Mcp-Method": "tools/list"}
for label, extra in [("no Origin (non-browser client)", {}),
                     ("Origin http://localhost:5173", {"Origin": "http://localhost:5173"}),
                     ("Origin http://evil.example", {"Origin": "http://evil.example"}),
                     ("Host evil.example (rebinding)", {"Host": f"evil.example:{port}"})]:
    r = web.post(URL, json=list_body, headers={**lh, **extra})
    print(f"{label:<32} -> HTTP {r.status_code} {r.text[:40] if r.status_code != 200 else ''}")

### 7. Cancel by closing the stream
Start a long call that streams progress, read two events, then **close the connection**. No `notifications/cancelled` is sent: the disconnect itself is the signal.

In [ ]:
server_events.clear()
body = tools_call("slow", {"steps": 30}, meta={**ENVELOPE, "progressToken": "tok-2"})
with web.stream("POST", URL, json=body, headers={**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "slow"}) as r:
    received = 0
    for line in r.iter_lines():
        if line.startswith("data: "):
            received += 1
            print("got:", json.loads(line[6:])["params"]["message"])
            if received == 2:
                break                    # leaving the block closes the response stream
time.sleep(0.5)
print("server side:", server_events)

### 8. Older clients and the high-level `Client`
How a 2026-07-28 server treats legacy traffic, then what the SDK `Client` sends on its own.

In [ ]:
print("GET  /mcp                ->", web.get(URL, headers=HEADERS).status_code, "(no standalone SSE stream any more)")
print("DELETE /mcp              ->", web.delete(URL, headers=HEADERS).status_code, "(no sessions to delete)")
note = {"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": 1}}
print("POST a notification      ->", web.post(URL, json=note, headers={**HEADERS, "Mcp-Method": "notifications/cancelled"}).status_code)
r = web.post(URL, json=list_body, headers={k: v for k, v in lh.items() if k != "MCP-Protocol-Version"})
print("POST without version hdr ->", r.status_code, r.json()["error"]["message"],
      "\n   (this SDK is dual-era: no header = an old client, so the legacy session transport answered)")

Finally, the real client. It discovers, sends `Mcp-Method`/`Mcp-Name`, and mirrors `x-mcp-header` arguments. If it does not know the schema yet, the server's `-32020` makes it re-list the tools and retry, exactly as the spec suggests.

In [ ]:
from mcp import Client

seen_headers.clear()


async def on_progress(progress, total, message):
    print(f"   progress {progress:.0f}/{total:.0f} {message}")


async with Client(URL) as client:
    print("negotiated:", client.protocol_version)
    print((await client.call_tool("run_sql", {"region": "Zürich", "query": "select 1"})).content[0].text)
    print((await client.call_tool("slow", {"steps": 2}, progress_callback=on_progress)).content[0].text)

print("\nwhat went over the wire:")
for h in seen_headers:
    print("  ", h)
uv.should_exit = True

### Your turn
1. Add `print("hello")` (to **stdout**) inside the `add` tool in section 1's server, restart it, and call `add`. What does the raw client see, and what does `Client` do? Then fix it with `file=sys.stderr`.
2. Add an `x-mcp-header` on a `number` parameter, or on a property inside an array's `items`, and call `list_tools()` from `Client(URL)`. The spec says HTTP clients must drop such tools: does this SDK?
3. Write a function `cancel_after(seconds)` that streams a `slow` call and closes it after a time budget. Compare the server-side event with section 2's stdio cancellation. Which side knows the reason for the cancel in each case?
4. Rebuild the HTTP app with `web_server.streamable_http_app(json_response=True)`. What happens to progress notifications, and why might a deployment behind a buffering proxy choose this anyway?